[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/83_xab_ali260509_solution.ipynb)

# Solution: xab (Max XOR Pair Count)

Reference solution.

## 解析

**结论：从高位到低位做数位 DP。用 16 个状态记录 `a`、`b` 是否仍分别贴着各自的上下界（4 个 tight 标志），每一位贪心地优先让异或位为 1，累加满足贪心选择的方案数。最终 `sum(f)` 即最优对数。**

### 状态
`a` 的取值受 `[la, ra]` 约束、`b` 受 `[lb, rb]` 约束。逐位确定 `a`、`b` 的这一位时，是否还“贴着”某个边界决定了这一位能取的范围。用 4 个布尔（贴 la / 贴 ra / 贴 lb / 贴 rb）压成 `0..15` 的掩码 `s`，`f[s]` = 处于该 tightness 且与当前贪心前缀一致的方案数。初始 `f[15]=1`（四个界都贴着，尚未选任何位）。

### 转移与贪心
对当前位 `pos`，枚举 `a` 的位 `ab`、`b` 的位 `bb`：
- 若贴着下界 `la` 则 `ab` 不能小于 `la` 的该位；贴着上界 `ra` 则不能更大；`b` 同理。
- 计算新 tightness（只有原来贴着且这一位恰好等于界位时才继续贴着）。
- 异或位 `cur = x_bit ^ ab ^ bb`。

因为要最大化异或值，高位优先：**只要存在能让本位为 1 的转移，就丢弃所有本位为 0 的转移**（`has_one` 分支），否则本位只能是 0。这样低位继续在“已确定的高位最优”前提下计数。

### 复杂度
31 位 × 16 状态 × 4 种 (ab, bb) 组合 = 常数级每题，总 `O(31 * 16 * 4)`。已用小范围 `O(区间²)` 暴力在数千组随机数据上对拍。

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass

In [ ]:
# no imports needed

In [ ]:
# ✅ SOLUTION

class Solution:
    def count_best_pairs(self, x, la, ra, lb, rb):
        f = [0] * 16
        f[15] = 1                      # start: tight on all 4 bounds
        for pos in range(30, -1, -1):
            xb = (x >> pos) & 1
            la_b = (la >> pos) & 1; ra_b = (ra >> pos) & 1
            lb_b = (lb >> pos) & 1; rb_b = (rb >> pos) & 1
            nz = [0] * 16; no = [0] * 16; has_one = False
            for s in range(16):
                c = f[s]
                if not c:
                    continue
                eq_la = s & 1; eq_ra = (s >> 1) & 1; eq_lb = (s >> 2) & 1; eq_rb = (s >> 3) & 1
                for ab in (0, 1):
                    if eq_la and ab < la_b:
                        continue
                    if eq_ra and ab > ra_b:
                        continue
                    n_la = 1 if (eq_la and ab == la_b) else 0
                    n_ra = 1 if (eq_ra and ab == ra_b) else 0
                    for bb in (0, 1):
                        if eq_lb and bb < lb_b:
                            continue
                        if eq_rb and bb > rb_b:
                            continue
                        n_lb = 1 if (eq_lb and bb == lb_b) else 0
                        n_rb = 1 if (eq_rb and bb == rb_b) else 0
                        ns = n_la | (n_ra << 1) | (n_lb << 2) | (n_rb << 3)
                        if xb ^ ab ^ bb == 1:
                            no[ns] += c; has_one = True
                        else:
                            nz[ns] += c
            f = no if has_one else nz    # greedily prefer a 1-bit in the XOR
        return sum(f)

In [ ]:
# Demo
sol = Solution()
print(sol.count_best_pairs(0, 1, 2, 0, 2))   # 2
print(sol.count_best_pairs(5, 1, 7, 6, 9))   # 2
print(sol.count_best_pairs(123456789, 0, 10**9, 0, 10**9))

In [ ]:
from torch_judge import check
check('xab_ali260509')